# 체인(Chain)이란?

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요:

1. 체이닝(Chaining) 개념을 이해하고 `PromptTemplate | Model | OutputParser` 구조를 설명할 수 있어요
2. 파이프(`|`) 연산자로 체인을 구성하고 `invoke()`로 실행할 수 있어요
3. `RunnableLambda`로 일반 Python 함수를 체인에 삽입할 수 있어요
4. 여러 LLM 호출을 연결하는 다단계 체인을 만들 수 있어요

## 사전 지식

- `01_LangChain Quick Start.ipynb`의 `invoke()`, `AIMessage` 개념
- Python 함수, 람다 표현식 기초

## 이전 노트북 연결

`01_LangChain Quick Start.ipynb`에서 모델에 직접 문자열이나 메시지 객체를 전달했어요. 이제 프롬프트 템플릿(Prompt Template)과 출력 파서(Output Parser)를 파이프 연산자로 연결해 재사용 가능한 체인을 만들어볼게요.

아래 다이어그램은 이 노트북에서 다룰 체인 흐름을 보여줘요.

```mermaid
flowchart LR
    A["딕셔너리 입력<br/>{topic: ...}"]:::input --> B["PromptTemplate<br/>템플릿 채우기"]:::process
    B --> C["ChatOpenAI<br/>LLM 호출"]:::process
    C --> D["StrOutputParser<br/>텍스트 추출"]:::process
    D --> E["RunnableLambda<br/>커스텀 변환"]:::process
    E --> F["최종 출력<br/>문자열 또는 딕셔너리"]:::output

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef error fill:#f8d7da,stroke:#dc3545,color:#721c24
    classDef storage fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e
    classDef external fill:#d1ecf1,stroke:#17a2b8,color:#0c5460
```

체이닝은 여러 작업을 연속으로 연결해 실행하는 방식이에요. 한 작업의 결과가 다음 작업의 입력으로 이어지므로, 복잡한 워크플로우(Workflow)를 간결하게 표현할 수 있어요.

In [ ]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda

load_dotenv()

# 모델 초기화
# temperature 미지정 시 기본값 사용 (적당한 창의성)
model = ChatOpenAI(model="gpt-4o-mini")

## 1. 기본적인 체이닝: PromptTemplate + OutputParser

`PromptTemplate`과 `StrOutputParser(String Output Parser)`를 파이프(`|`) 연산자로 연결해 첫 번째 체인을 만들어볼게요.

- `PromptTemplate.from_template()`: `{변수명}` 형식의 플레이스홀더를 자동으로 인식해요
- `StrOutputParser`: `AIMessage`에서 `.content` 문자열만 추출해요
- 파이프 연산자: 왼쪽 출력이 오른쪽 입력으로 전달돼요

In [ ]:
# ---------------------------------------------------
# PromptTemplate + Model + OutputParser 체인 구성하기
# ---------------------------------------------------
# ============================================================
# TODO: {question} 자리에 원하는 질문이 들어가도록 invoke()를 호출해요
# 힌트: invoke()에 딕셔너리 {"question": "..."} 형태로 전달해요
# 예상 결과: 질문에 대한 LLM 응답 문자열이 출력돼요
# ============================================================


## 2. 여러 단계를 거치는 체이닝 (PromptTemplate + RunnableLambda)

PromptTemplate과 RunnableLambda를 사용하여 여러 단계를 파이프로 연결합니다.

> **한계점**: 아래 예제에서는 `add_topic_info` 함수 내에 `"파이썬"`이 하드코딩되어 있습니다. 파이프라인 중간에서 원본 입력값을 유지하려면 `RunnablePassthrough.assign()`이나 `RunnableParallel`을 사용해야 합니다. 이 패턴은 `04_Runnable_Basic.ipynb`에서 자세히 다룹니다.

In [ ]:
# ---------------------------------------------------
# RunnableLambda로 중간 변환 함수 삽입하기
# ---------------------------------------------------


## 3. Runnable 체인을 함수로 만들기

앞서 `RunnableLambda`로 중간 변환을 삽입했어요. 체인 생성 로직 자체를 함수로 캡슐화하면 여러 곳에서 재사용하기가 편해요.

다음 셀에서 체인 생성 함수를 만들고 호출해볼게요.

In [ ]:
# ---------------------------------------------------
# 체인 생성 로직을 함수로 캡슐화하기
# ---------------------------------------------------
# ============================================================
# TODO: create_explanation_chain()을 호출하고 다른 topic으로 실행해봐요
# 힌트: chain.invoke({"topic": "원하는 주제"}) 형태로 호출해요
# 예상 결과: 입력 주제에 대한 설명 문자열이 출력돼요
# ============================================================


## 4. 파이프 연산자(`|`)를 사용한 체이닝

재사용 가능한 체인 함수를 만들어봤어요. 이제 파이프 연산자가 기존 방식과 어떻게 다른지 직접 비교해볼게요.

파이프 연산자를 사용하면 중간 변수 없이 `PromptTemplate | model | StrOutputParser()` 형태로 한 줄에 표현할 수 있어요. 데이터가 왼쪽에서 오른쪽으로 흐르는 방향이 코드에서 바로 보여요.

In [ ]:
# ---------------------------------------------------
# 파이프 없이 각 단계를 수동으로 실행하는 방식 (비교용)
# ---------------------------------------------------
# 각 단계의 출력을 중간 변수에 담아 다음 단계로 전달해야 해요


파이프 연산자를 사용하면 위의 코드를 더 간결하게 작성할 수 있습니다:


In [ ]:
# ---------------------------------------------------
# 파이프 연산자로 체인을 한 줄에 표현하기
# ---------------------------------------------------
# ============================================================
# TODO: summary_prompt 변수를 정의하고 파이프로 chain을 구성해요
# 힌트: PromptTemplate.from_template("{text}") | model | StrOutputParser()
# 예상 결과: 입력 텍스트를 한 문장으로 요약한 결과가 출력돼요
# ============================================================


### 파이프 연산자의 동작 원리

파이프 연산자(`|`)는 왼쪽의 Runnable 결과를 오른쪽 Runnable의 입력으로 전달합니다:
- `PromptTemplate | model` → 프롬프트 결과를 model에 전달
- `model | OutputParser` → model의 결과를 OutputParser에 전달
- `PromptTemplate | model | OutputParser` → 순차적으로 연결

데이터가 왼쪽에서 오른쪽으로 흐르는 것을 시각적으로 표현할 수 있어 코드가 더 읽기 쉬워집니다.

> **참고**: LangChain에서는 `__or__` 메서드를 오버라이드하여 `|` 연산자로 체인을 연결할 수 있어요. 이는 특정 Python 버전에 의존하지 않으며, LangChain이 지원하는 모든 Python 버전에서 동작합니다.


In [ ]:
# ---------------------------------------------------
# RunnableLambda로 후처리 함수를 체인에 추가하기
# ---------------------------------------------------


### 파이프 연산자의 장점

- **가독성**: 코드의 흐름이 왼쪽에서 오른쪽으로 자연스럽게 읽힙니다
  - `입력 → 처리1 → 처리2 → 출력` 형태로 데이터 흐름을 직관적으로 표현
- **간결성**: 중간 변수를 줄여 코드가 더 짧아집니다
- **연결성**: 여러 단계를 쉽게 연결할 수 있습니다
  - 새로운 단계를 추가할 때 `| 새로운단계`만 붙이면 됩니다


In [ ]:
# ---------------------------------------------------
# 파이프 사용 전·후 코드 비교하기
# ---------------------------------------------------


## 5. 여러 LLM 호출을 연결하는 체인

파이프 연산자의 장점을 확인했어요. 이제 한 LLM의 출력을 다음 LLM의 입력으로 연결하는 다단계 체인을 만들어볼게요.

첫 번째 체인이 설명을 생성하고, 두 번째 체인이 그 설명을 요약해요. 두 체인을 하나로 합칠 때 `RunnableLambda`로 데이터 형식(문자열 → 딕셔너리)을 변환하는 방법도 확인할 수 있어요.

In [ ]:
# ---------------------------------------------------
# 두 LLM 호출을 연결하는 다단계 체인 만들기
# ---------------------------------------------------
# ============================================================
# TODO: explanation_chain의 출력을 summary_chain의 입력으로 연결해요
# 힌트: RunnableLambda(lambda text: {"text": text})로 문자열→딕셔너리 변환
# 예상 결과: "파이썬"에 대한 설명을 생성하고, 그것을 다시 한 문장으로 요약해요
# ============================================================


In [ ]:
# ---------------------------------------------------
# RunnableLambda로 요약 프롬프트를 문자열로 직접 생성하기
# ---------------------------------------------------


## 6. OutputParser를 활용한 구조화된 출력

다단계 체인을 다뤄봤어요. 이번에는 LLM 응답을 딕셔너리 형태로 파싱하는 커스텀 파서를 만들어볼게요.

`StrOutputParser`는 단순 문자열만 반환해요. 더 복잡한 구조가 필요할 때는 `RunnableLambda`로 파싱 함수를 체인에 연결할 수 있어요.

> **실무 팁**: 프로덕션 환경에서는 LLM 출력 형식이 일정하지 않을 수 있어요. `PydanticOutputParser`나 `JsonOutputParser` 같은 견고한 파서를 사용하는 편이 안정적이에요. 이 내용은 이후 노트북에서 다룰 거예요.

In [ ]:
# ---------------------------------------------------
# RunnableLambda로 LLM 응답을 구조화된 딕셔너리로 파싱하기
# ---------------------------------------------------


## 핵심 요약

이 노트북에서 다음 내용을 배웠어요:

- **`PromptTemplate`**: `{변수명}` 플레이스홀더로 재사용 가능한 프롬프트를 만들어요
- **`StrOutputParser`**: `AIMessage`에서 텍스트 문자열만 추출해요
- **`RunnableLambda`**: 일반 Python 함수를 파이프에 연결할 수 있는 Runnable(러너블)로 변환해요
- **파이프 연산자(`|`)**: 여러 Runnable을 연결해 데이터 흐름을 직관적으로 표현해요
- **다단계 체인**: `RunnableLambda`로 체인 간 데이터 형식을 변환해 여러 LLM 호출을 연결할 수 있어요

## 다음 노트북 예고

다음 `03_LCEL_basic.ipynb`에서는 LangChain Expression Language(LCEL)의 표준 인터페이스인 `invoke`, `batch`, `stream`과 비동기 메서드를 더 깊이 다뤄요. 체인을 다양한 방식으로 실행하는 법을 알아볼게요.